In [36]:
import pandas as pd
import numpy as np
import nltk
import re

In [37]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
val = pd.read_csv('val.csv')

In [38]:
def clean_english(text):
    text = str(text).lower()                       
    text = re.sub(r"[^a-z0-9\s\.\,\!\?\'\"]-", " ", text)  
    text = re.sub(r'\s+', ' ', text).strip()       
    return text

def clean_hindi(text): # kept punctuation for hindi
    text = str(text)
    text = re.sub(r'[^\u0900-\u097F\s\.\,\!\?।]', ' ', text) 
    text = re.sub(r'\s+', ' ', text).strip()        
    return text

# Apply to all 3 sets
for df in [train, val, test]:
    df['english'] = df['english'].apply(clean_english)
    df['hindi']   = df['hindi'].apply(clean_hindi)

print("Cleaning done!")
print(train[['english','hindi']].head(3))

Cleaning done!
                                             english                                              hindi
0  shri m.p chaudhari, director (finance) and oth...  चौधरी, निदेशक वित्त और मॉयल लिमिटेड के अन्य वर...
1  the court has already granted them interim rel...  कोर्ट ने फिलहाल उन्हें गिरफ्तारी से अंतरिम संर...
2              this is the biggest advantage for us.               सबसे बड़ा फायदा हमारे लिए तो यही है.


In [39]:
from indicnlp.tokenize import indic_tokenize

def tokenize_english(text):
    return nltk.word_tokenize(text) 

def tokenize_hindi(text):
    return indic_tokenize.trivial_tokenize(text, lang='hi')  

# Test on one example first
sample_en = train['english'][0]
sample_hi = train['hindi'][0]

print("English sentence:", sample_en)
print("English tokens:  ", tokenize_english(sample_en))
print()
print("Hindi sentence:", sample_hi)
print("Hindi tokens:  ", tokenize_hindi(sample_hi))

English sentence: shri m.p chaudhari, director (finance) and other senior officials of moil limited were also present.
English tokens:   ['shri', 'm.p', 'chaudhari', ',', 'director', '(', 'finance', ')', 'and', 'other', 'senior', 'officials', 'of', 'moil', 'limited', 'were', 'also', 'present', '.']

Hindi sentence: चौधरी, निदेशक वित्त और मॉयल लिमिटेड के अन्य वरिष्ठ अधिकारीगण भी इस अवसर पर उपस्थित थे।
Hindi tokens:   ['चौधरी', ',', 'निदेशक', 'वित्त', 'और', 'मॉयल', 'लिमिटेड', 'के', 'अन्य', 'वरिष्ठ', 'अधिकारीगण', 'भी', 'इस', 'अवसर', 'पर', 'उपस्थित', 'थे', '।']


In [40]:
# applying tokenisation to all 3 files

from tqdm import tqdm
tqdm.pandas()  # shows a progress bar

print("Tokenizing training set...")
train['eng_tokens'] = train['english'].progress_apply(tokenize_english)
train['hi_tokens']  = train['hindi'].progress_apply(tokenize_hindi)

print("Tokenizing validation set...")
val['eng_tokens'] = val['english'].progress_apply(tokenize_english)
val['hi_tokens']  = val['hindi'].progress_apply(tokenize_hindi)

print("Tokenizing test set...")
test['eng_tokens'] = test['english'].progress_apply(tokenize_english)
test['hi_tokens']  = test['hindi'].progress_apply(tokenize_hindi)

print("\nDone! Sample:")
print(train[['eng_tokens','hi_tokens']].head(2))

Tokenizing training set...


  0%|          | 0/120000 [00:00<?, ?it/s]

100%|██████████| 120000/120000 [00:00<00:00, 131055.31it/s]


Tokenizing validation set...


100%|██████████| 15000/15000 [00:00<00:00, 133841.40it/s]


Tokenizing test set...


100%|██████████| 15000/15000 [00:00<00:00, 18090.28it/s]


Done! Sample:
                                          eng_tokens                                          hi_tokens
0  [shri, m.p, chaudhari, ,, director, (, finance...  [चौधरी, ,, निदेशक, वित्त, और, मॉयल, लिमिटेड, क...
1  [the, court, has, already, granted, them, inte...  [कोर्ट, ने, फिलहाल, उन्हें, गिरफ्तारी, से, अंत...


In [41]:
# adding special tokens

def add_special_tokens(tokens):
    return ['<SOS>'] + tokens + ['<EOS>']

train['eng_tokens'] = train['eng_tokens'].apply(add_special_tokens)
train['hi_tokens']  = train['hi_tokens'].apply(add_special_tokens)

val['eng_tokens'] = val['eng_tokens'].apply(add_special_tokens)
val['hi_tokens']  = val['hi_tokens'].apply(add_special_tokens)

test['eng_tokens'] = test['eng_tokens'].apply(add_special_tokens)
test['hi_tokens']  = test['hi_tokens'].apply(add_special_tokens)

print(train['eng_tokens'][0])
print(train['hi_tokens'][0])

['<SOS>', 'shri', 'm.p', 'chaudhari', ',', 'director', '(', 'finance', ')', 'and', 'other', 'senior', 'officials', 'of', 'moil', 'limited', 'were', 'also', 'present', '.', '<EOS>']
['<SOS>', 'चौधरी', ',', 'निदेशक', 'वित्त', 'और', 'मॉयल', 'लिमिटेड', 'के', 'अन्य', 'वरिष्ठ', 'अधिकारीगण', 'भी', 'इस', 'अवसर', 'पर', 'उपस्थित', 'थे', '।', '<EOS>']


In [42]:
# Building Vocabulary

from collections import Counter

def build_vocab(token_lists, min_freq=2):
    # Count frequency of every word
    counter = Counter()
    for tokens in token_lists:
        counter.update(tokens)
    
    # Start with special tokens
    vocab = {
        '<PAD>': 0,
        '<SOS>': 1,
        '<EOS>': 2,
        '<UNK>': 3
    }
    
    # Add words that appear at least min_freq times
    for word, freq in counter.items():
        if freq >= min_freq and word not in vocab:
            vocab[word] = len(vocab)
    
    return vocab

# Build from TRAINING SET ONLY
eng_vocab = build_vocab(train['eng_tokens'], min_freq=5)
hi_vocab  = build_vocab(train['hi_tokens'],  min_freq=5)

print(f"English vocabulary size: {len(eng_vocab):,}")
print(f"Hindi vocabulary size:   {len(hi_vocab):,}")

English vocabulary size: 18,671
Hindi vocabulary size:   18,715


In [43]:
#  Check a few words are in vocab correctly to verify

print(eng_vocab['<PAD>'], eng_vocab['<SOS>'], eng_vocab['<EOS>'], eng_vocab['<UNK>'])  
print(hi_vocab['<PAD>'], hi_vocab['<SOS>'], hi_vocab['<EOS>'], hi_vocab['<UNK>'])   

# Checking a random word
print("Index of 'the':", eng_vocab.get('the', 'NOT FOUND'))
print("Index of 'और':", hi_vocab.get('और', 'NOT FOUND'))

0 1 2 3
0 1 2 3
Index of 'the': 20
Index of 'और': 8


In [44]:
import pickle

with open('eng_vocab.pkl', 'wb') as f:
    pickle.dump(eng_vocab, f)

with open('hi_vocab.pkl', 'wb') as f:
    pickle.dump(hi_vocab, f)

print("Vocabularies saved!")

Vocabularies saved!


In [45]:
# embeddingd

def tokens_to_indices(tokens, vocab):
    return [vocab.get(token, vocab['<UNK>']) for token in tokens]

# Apply to all 3 sets
for df in [train, val, test]:
    df['eng_indices'] = df['eng_tokens'].apply(lambda x: tokens_to_indices(x, eng_vocab))
    df['hi_indices']  = df['hi_tokens'].apply(lambda x: tokens_to_indices(x, hi_vocab))

# Verify
print("Tokens:  ", train['eng_tokens'][0])
print("Indices: ", train['eng_indices'][0])

Tokens:   ['<SOS>', 'shri', 'm.p', 'chaudhari', ',', 'director', '(', 'finance', ')', 'and', 'other', 'senior', 'officials', 'of', 'moil', 'limited', 'were', 'also', 'present', '.', '<EOS>']
Indices:  [1, 4, 3, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 3, 15, 16, 17, 18, 19, 2]


In [46]:
import fasttext

print("Loading FastText models... (takes a minute)")
ft_en = fasttext.load_model('cc.en.300.bin')
ft_hi = fasttext.load_model('cc.hi.300.bin')
print("Loaded!")

# Build English embedding matrix
eng_matrix = np.zeros((len(eng_vocab), 300))
for word, idx in eng_vocab.items():
    eng_matrix[idx] = ft_en.get_word_vector(word)

# Build Hindi embedding matrix
hi_matrix = np.zeros((len(hi_vocab), 300))
for word, idx in hi_vocab.items():
    hi_matrix[idx] = ft_hi.get_word_vector(word)

print(f"English matrix shape: {eng_matrix.shape}")
print(f"Hindi matrix shape:   {hi_matrix.shape}")

Loading FastText models... (takes a minute)
Loaded!


English matrix shape: (18671, 300)
Hindi matrix shape:   (18715, 300)


In [48]:
np.save('eng_embedding_matrix.npy', eng_matrix)
np.save('hi_embedding_matrix.npy', hi_matrix)

print("Embedding matrices saved!")
print(f"English matrix size: {eng_matrix.nbytes / 1024 / 1024:.1f} MB")
print(f"Hindi matrix size:   {hi_matrix.nbytes / 1024 / 1024:.1f} MB")

Embedding matrices saved!
English matrix size: 42.7 MB
Hindi matrix size:   42.8 MB


In [49]:
# padding

# Analyze lengths to find MAX_LEN
eng_lengths = train['eng_indices'].apply(len)
hi_lengths  = train['hi_indices'].apply(len)

print("English lengths:")
print(eng_lengths.describe())
print("\nHindi lengths:")
print(hi_lengths.describe())

print("\n95th percentile English:", int(eng_lengths.quantile(0.95)))
print("95th percentile Hindi:  ", int(hi_lengths.quantile(0.95)))

English lengths:
count    120000.000000
mean         19.846425
std          11.084191
min           5.000000
25%          11.000000
50%          17.000000
75%          26.000000
max          88.000000
Name: eng_indices, dtype: float64

Hindi lengths:
count    120000.000000
mean         20.387650
std          11.208827
min           3.000000
25%          12.000000
50%          17.000000
75%          26.000000
max          84.000000
Name: hi_indices, dtype: float64

95th percentile English: 42
95th percentile Hindi:   43


In [50]:
MAX_LEN = 50

def pad_sequence(indices, max_len):
    if len(indices) >= max_len:
        return indices[:max_len]      
    else:
        return indices + [0] * (max_len - len(indices))  # pad with 0s

# Apply to all 3 sets
for df in [train, val, test]:
    df['eng_padded'] = df['eng_indices'].apply(lambda x: pad_sequence(x, MAX_LEN))
    df['hi_padded']  = df['hi_indices'].apply(lambda x: pad_sequence(x, MAX_LEN))

# Verify
print("Length of padded sequence:", len(train['eng_padded'][0]))
print(train['eng_padded'][0])

Length of padded sequence: 50
[1, 4, 3, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 3, 15, 16, 17, 18, 19, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [51]:
train.to_pickle('train_processed.pkl')
val.to_pickle('val_processed.pkl')
test.to_pickle('test_processed.pkl')

print("Saved!")
print("\nFinal columns in each dataframe:")
print(train.columns.tolist())

Saved!

Final columns in each dataframe:
['english', 'hindi', 'eng_tokens', 'hi_tokens', 'eng_indices', 'hi_indices', 'eng_padded', 'hi_padded']


In [52]:
train.head()

,english,hindi,eng_tokens,hi_tokens,eng_indices,hi_indices,eng_padded,hi_padded
0,"shri m.p chaudhari, director (finance) and oth...","चौधरी, निदेशक वित्त और मॉयल लिमिटेड के अन्य वर...","[<SOS>, shri, m.p, chaudhari, ,, director, (, ...","[<SOS>, चौधरी, ,, निदेशक, वित्त, और, मॉयल, लिम...","[1, 4, 3, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 4, 5, 6, 7, 8, 3, 9, 10, 11, 12, 13, 14, 1...","[1, 4, 3, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 4, 5, 6, 7, 8, 3, 9, 10, 11, 12, 13, 14, 1..."
1,the court has already granted them interim rel...,कोर्ट ने फिलहाल उन्हें गिरफ्तारी से अंतरिम संर...,"[<SOS>, the, court, has, already, granted, the...","[<SOS>, कोर्ट, ने, फिलहाल, उन्हें, गिरफ्तारी, ...","[1, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 19...","[1, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31...","[1, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 19...","[1, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31..."
2,this is the biggest advantage for us.,सबसे बड़ा फायदा हमारे लिए तो यही है.,"[<SOS>, this, is, the, biggest, advantage, for...","[<SOS>, सबसे, बड़ा, फायदा, हमारे, लिए, तो, यही...","[1, 30, 31, 20, 32, 33, 34, 35, 19, 2]","[1, 33, 34, 35, 36, 37, 38, 39, 31, 32, 2]","[1, 30, 31, 20, 32, 33, 34, 35, 19, 2, 0, 0, 0...","[1, 33, 34, 35, 36, 37, 38, 39, 31, 32, 2, 0, ..."
3,gunfire erupted after the explosion.,धमाके के बाद आग लग गयी।,"[<SOS>, gunfire, erupted, after, the, explosio...","[<SOS>, धमाके, के, बाद, आग, लग, गयी, ।, <EOS>]","[1, 3, 36, 37, 20, 38, 19, 2]","[1, 40, 10, 41, 42, 43, 44, 20, 2]","[1, 3, 36, 37, 20, 38, 19, 2, 0, 0, 0, 0, 0, 0...","[1, 40, 10, 41, 42, 43, 44, 20, 2, 0, 0, 0, 0,..."
4,"so, b1 is between a0 and a1 in size, with an a...",इसीलिये और के बीच का आकार है जिसका क्षेत्रफल ....,"[<SOS>, so, ,, b1, is, between, a0, and, a1, i...","[<SOS>, इसीलिये, और, के, बीच, का, आकार, है, जि...","[1, 39, 5, 3, 31, 40, 3, 10, 3, 41, 42, 5, 43,...","[1, 3, 8, 10, 45, 46, 47, 31, 48, 49, 32, 50, ...","[1, 39, 5, 3, 31, 40, 3, 10, 3, 41, 42, 5, 43,...","[1, 3, 8, 10, 45, 46, 47, 31, 48, 49, 32, 50, ..."


In [53]:
# See one complete example clearly
print("English sentence:  ", train['english'][0])
print("Hindi sentence:    ", train['hindi'][0])
print("English tokens:    ", train['eng_tokens'][0])
print("Hindi tokens:      ", train['hi_tokens'][0])
print("English indices:   ", train['eng_indices'][0])
print("Hindi indices:     ", train['hi_indices'][0])
print("English padded:    ", train['eng_padded'][0])
print("Hindi padded:      ", train['hi_padded'][0])

English sentence:   shri m.p chaudhari, director (finance) and other senior officials of moil limited were also present.
Hindi sentence:     चौधरी, निदेशक वित्त और मॉयल लिमिटेड के अन्य वरिष्ठ अधिकारीगण भी इस अवसर पर उपस्थित थे।
English tokens:     ['<SOS>', 'shri', 'm.p', 'chaudhari', ',', 'director', '(', 'finance', ')', 'and', 'other', 'senior', 'officials', 'of', 'moil', 'limited', 'were', 'also', 'present', '.', '<EOS>']
Hindi tokens:       ['<SOS>', 'चौधरी', ',', 'निदेशक', 'वित्त', 'और', 'मॉयल', 'लिमिटेड', 'के', 'अन्य', 'वरिष्ठ', 'अधिकारीगण', 'भी', 'इस', 'अवसर', 'पर', 'उपस्थित', 'थे', '।', '<EOS>']
English indices:    [1, 4, 3, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 3, 15, 16, 17, 18, 19, 2]
Hindi indices:      [1, 4, 5, 6, 7, 8, 3, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2]
English padded:     [1, 4, 3, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 3, 15, 16, 17, 18, 19, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Hindi padded:       [1

In [54]:
val.head()

,english,hindi,eng_tokens,hi_tokens,eng_indices,hi_indices,eng_padded,hi_padded
0,i did not come to politics after retirement.,मैं संन्यास लेने के लिए राजनीति में नहीं आया हूं.,"[<SOS>, i, did, not, come, to, politics, after...","[<SOS>, मैं, संन्यास, लेने, के, लिए, राजनीति, ...","[1, 67, 141, 59, 905, 75, 3782, 37, 6060, 19, 2]","[1, 316, 3422, 638, 10, 37, 3803, 67, 61, 2179...","[1, 67, 141, 59, 905, 75, 3782, 37, 6060, 19, ...","[1, 316, 3422, 638, 10, 37, 3803, 67, 61, 2179..."
1,there were numerous attempts to give the murde...,हत्या को राजनीतिक लाभ के लिए एक सांप्रदायिक को...,"[<SOS>, there, were, numerous, attempts, to, g...","[<SOS>, हत्या, को, राजनीतिक, लाभ, के, लिए, एक,...","[1, 378, 16, 6846, 8888, 75, 60, 20, 1419, 90,...","[1, 123, 81, 1423, 2156, 10, 37, 64, 8333, 130...","[1, 378, 16, 6846, 8888, 75, 60, 20, 1419, 90,...","[1, 123, 81, 1423, 2156, 10, 37, 64, 8333, 130..."
2,it was explained that the nss had been conduct...,यह बताया गया था कि एनएसएस यह सर्वे सेवा क्षेत्...,"[<SOS>, it, was, explained, that, the, nss, ha...","[<SOS>, यह, बताया, गया, था, कि, एनएसएस, यह, सर...","[1, 47, 138, 5191, 146, 20, 12609, 436, 346, 1...","[1, 170, 1056, 200, 135, 239, 11105, 170, 3758...","[1, 47, 138, 5191, 146, 20, 12609, 436, 346, 1...","[1, 170, 1056, 200, 135, 239, 11105, 170, 3758..."
3,we have no dispute because land to be cultivat...,गांव के ही एक किसान सुनील गुज्जर ने कहा कि हमा...,"[<SOS>, we, have, no, dispute, because, land, ...","[<SOS>, गांव, के, ही, एक, किसान, सुनील, गुज्जर...","[1, 50, 66, 988, 132, 202, 2443, 75, 84, 16044...","[1, 1150, 10, 426, 64, 5943, 6575, 3, 22, 238,...","[1, 50, 66, 988, 132, 202, 2443, 75, 84, 16044...","[1, 1150, 10, 426, 64, 5943, 6575, 3, 22, 238,..."
4,mohammed shami will lead the attack and navdee...,ऐसे में मोहम्मद शमी तेज गेंदबाजी आक्रमण की अगु...,"[<SOS>, mohammed, shami, will, lead, the, atta...","[<SOS>, ऐसे, में, मोहम्मद, शमी, तेज, गेंदबाजी,...","[1, 3757, 4818, 49, 2719, 20, 2423, 10, 4160, ...","[1, 993, 67, 3778, 10725, 2347, 4347, 7242, 10...","[1, 3757, 4818, 49, 2719, 20, 2423, 10, 4160, ...","[1, 993, 67, 3778, 10725, 2347, 4347, 7242, 10..."


In [55]:
test.head()

,english,hindi,eng_tokens,hi_tokens,eng_indices,hi_indices,eng_padded,hi_padded
0,a pakistani court has constituted a two-member...,पाकिस्तान की एक अदालत ने अल अज़ी़जि़या भ्रष्टा...,"[<SOS>, a, pakistani, court, has, constituted,...","[<SOS>, पाकिस्तान, की, एक, अदालत, ने, अल, अज़ी...","[1, 90, 845, 21, 22, 4891, 90, 3, 2997, 75, 30...","[1, 439, 104, 64, 1225, 22, 2143, 3, 2601, 334...","[1, 90, 845, 21, 22, 4891, 90, 3, 2997, 75, 30...","[1, 439, 104, 64, 1225, 22, 2143, 3, 2601, 334..."
1,the explosion caused panic in the area.,इस विस्फोट के चलते क्षेत्र में हड़कंप मच गया।,"[<SOS>, the, explosion, caused, panic, in, the...","[<SOS>, इस, विस्फोट, के, चलते, क्षेत्र, में, ह...","[1, 20, 38, 3850, 10280, 41, 20, 45, 19, 2]","[1, 15, 2331, 10, 1463, 431, 67, 2636, 3522, 2...","[1, 20, 38, 3850, 10280, 41, 20, 45, 19, 2, 0,...","[1, 15, 2331, 10, 1463, 431, 67, 2636, 3522, 2..."
2,government approved this,सरकार ने दी मंजूरी,"[<SOS>, government, approved, this, <EOS>]","[<SOS>, सरकार, ने, दी, मंजूरी, <EOS>]","[1, 210, 1413, 30, 2]","[1, 428, 22, 124, 4437, 2]","[1, 210, 1413, 30, 2, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 428, 22, 124, 4437, 2, 0, 0, 0, 0, 0, 0, 0..."
3,india has a population of 130 crore.,"भारत, मन बना चुका है, भारत के करोड़ लोग मन बना...","[<SOS>, india, has, a, population, of, 130, cr...","[<SOS>, भारत, ,, मन, बना, चुका, है, ,, भारत, क...","[1, 348, 22, 90, 798, 14, 7753, 1458, 19, 2]","[1, 328, 5, 4807, 56, 1808, 31, 5, 328, 10, 14...","[1, 348, 22, 90, 798, 14, 7753, 1458, 19, 2, 0...","[1, 328, 5, 4807, 56, 1808, 31, 5, 328, 10, 14..."
4,that is why he has more affinity with the bjp.,सहयोगी होने के कारण उसका भाजपा से ज्यादा प्रेम...,"[<SOS>, that, is, why, he, has, more, affinity...","[<SOS>, सहयोगी, होने, के, कारण, उसका, भाजपा, स...","[1, 146, 31, 56, 189, 22, 1364, 3, 43, 20, 257...","[1, 3574, 448, 10, 304, 1651, 241, 26, 341, 47...","[1, 146, 31, 56, 189, 22, 1364, 3, 43, 20, 257...","[1, 3574, 448, 10, 304, 1651, 241, 26, 341, 47..."


In [56]:
print("Vocab sizes:")
print(f"English: {len(eng_vocab):,}")
print(f"Hindi:   {len(hi_vocab):,}")

print("\nMatrix shapes:")
print(f"English: {eng_matrix.shape}")
print(f"Hindi:   {hi_matrix.shape}")

print("\nSample padded sequence:")
print(train['eng_padded'][0])
print(f"Length: {len(train['eng_padded'][0])}")

Vocab sizes:
English: 18,671
Hindi:   18,715

Matrix shapes:
English: (18671, 300)
Hindi:   (18715, 300)

Sample padded sequence:
[1, 4, 3, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 3, 15, 16, 17, 18, 19, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Length: 50
